In [1]:
%pip install folium pandas requests


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: Full Code Program Utama Visualisasi EAS
import folium
from folium import plugins
import json
import requests
import time
import os
import pandas as pd

# 1. Definisi Warna Rute per Klaster Wilayah (Kurir 1)
warna_wilayah = {
    "pusat": "#c0392b",   # Merah
    "utara": "#2980b9",   # Biru
    "selatan": "#27ae60", # Hijau
    "timur": "#8e44ad",   # Ungu
    "barat": "#d4ac0d"    # Kuning
}

# Warna khusus Kurir Genap / Kurir 2 (Tone lebih gelap agar mudah dibedakan)
warna_kurir_2 = {
    "pusat": "#78281F",   # Merah Gelap
    "utara": "#154360",   # Biru Gelap
    "selatan": "#145A32", # Hijau Gelap
    "timur": "#4A235A",   # Ungu Gelap
    "barat": "#7D6608"    # Coklat/Zaitun Gelap
}

# 2. Fungsi Load Dataset CSV dan Mapping Aturan Prioritas Kebijakan EAS
def load_koordinat_eas(path_csv):
    try:
        df = pd.read_csv(path_csv)
        df['Nama Puskesmas'] = df['Nama Puskesmas'].str.strip()
        info_pusk = {}
        
        for index, row in df.iterrows():
            nama = row['Nama Puskesmas']
            jaringan = str(row['Jaringan Pelayanan']).strip()
            layanan = str(row['Jenis Layanan']).strip()
            
            lat_str = str(row['Latitude']).replace(',', '').replace('"', '').strip()
            lon_str = str(row['Longitude']).replace(',', '').replace('"', '').strip()
            try:
                lat = float(lat_str[:2] + '.' + lat_str[2:])
                lon = float(lon_str[:3] + '.' + lon_str[3:])
            except:
                lat, lon = -7.32229, 112.7717 
            
            if nama == 'UPTD Gudang Farmasi':
                warna = 'black'
                prioritas = 'Depot Farmasi Utama'
            elif jaringan == 'Puskesmas Induk' and layanan == 'Rawat Inap':
                warna = '#e74c3c'  
                prioritas = 'Prioritas 1 (Induk - Rawat Inap)'
            elif jaringan == 'Puskesmas Induk' and layanan == 'Rawat Jalan':
                warna = '#e67e22'  
                prioritas = 'Prioritas 2 (Induk - Rawat Jalan)'
            else:
                warna = '#3498db'  
                prioritas = 'Prioritas 3 (Pembantu - Rawat Jalan)'
                
            info_pusk[nama] = {
                'koordinat': [lat, lon],
                'jaringan': jaringan,
                'layanan': layanan,
                'warna_prioritas': warna,
                'label_prioritas': prioritas
            }
        return info_pusk
    except Exception as e:
        print(f"Gagal membaca file CSV: {e}")
        return {}

# 3. Fungsi Memanggil API OSRM
def get_real_route_osrm(coord_asal, coord_tujuan):
    lon_asal, lat_asal = coord_asal[1], coord_asal[0]
    lon_tujuan, lat_tujuan = coord_tujuan[1], coord_tujuan[0]
    
    url = f"http://router.project-osrm.org/route/v1/driving/{lon_asal},{lat_asal};{lon_tujuan},{lat_tujuan}?overview=full&geometries=geojson"
    headers = {'User-Agent': 'EAS-Routing-App-Surabaya/1.0'}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        res = r.json()
        if res['code'] == 'Ok':
            geometry = res['routes'][0]['geometry']['coordinates']
            return [[point[1], point[0]] for point in geometry]
    except:
        pass
    return [coord_asal, coord_tujuan]

# 4. Fungsi Utama Pembuat Peta
def buat_peta_rute_eas():
    if os.path.exists('koordinat_eas.csv'):
        path_csv = 'koordinat_eas.csv'
    elif os.path.exists('../data/koordinat_eas.csv'):
        path_csv = '../data/koordinat_eas.csv'
    else:
        path_csv = '../koordinat_eas.csv' 
        
    info_pusk = load_koordinat_eas(path_csv)
    if not info_pusk:
        print("Proses dihentikan karena master data CSV tidak ditemukan.")
        return None

    nama_gudang = "UPTD Gudang Farmasi"
    if nama_gudang in info_pusk:
        titik_pusat = info_pusk[nama_gudang]['koordinat']
    else:
        titik_pusat = list(info_pusk.values())[0]['koordinat']

    peta = folium.Map(location=titik_pusat, zoom_start=12, tiles="CartoDB positron", zoom_control=False)
    plugins.LocateControl().add_to(peta)

    folium.Marker(
        location=titik_pusat,
        popup="<b>UPTD Gudang Farmasi Surabaya</b><br>Pusat Distribusi Utama",
        icon=folium.Icon(color='black', icon='building', prefix='fa')
    ).add_to(peta)

    algoritma_files = {
        "ga": {"nama": "Algoritma GA", "file_json": "rute_ga.json"},
        "ma": {"nama": "Algoritma MA", "file_json": "rute_ma.json"},
        "aco": {"nama": "Algoritma ACO", "file_json": "rute_aco.json"},
        "pso": {"nama": "Algoritma PSO", "file_json": "rute_dpso.json"}
    }

    rute_per_algo_wilayah = {"ga": {}, "ma": {}, "aco": {}, "pso": {}}
    
    print("Memulai pembacaan seluruh JSON hasil algoritma dan penarikan jalan OSRM. Mohon tunggu, proses ini memakan waktu agar server peta tidak memblokir koneksi...")

    for id_algo, info in algoritma_files.items():
        filepath = info["file_json"]
        
        if not os.path.exists(filepath):
            filepath = '../output_json/' + info["file_json"]
            if not os.path.exists(filepath):
                continue
                
        with open(filepath, 'r') as f:
            data_json = json.load(f)
            
        for wilayah, data_wilayah in data_json['hasil_per_klaster'].items():
            wilayah_lower_key = wilayah.lower().strip()
            rute_per_algo_wilayah[id_algo][wilayah_lower_key] = []
            
            for kurir in data_wilayah.get('rute_per_kurir', []):
                id_kurir = kurir['id_kurir']
                jarak_km = kurir.get('jarak_tempuh_km', 0)
                waktu_menit = kurir.get('waktu_tempuh_menit', 0)
                urutan_lokasi = kurir['urutan_kunjungan']
                koordinat_lokasi = kurir['koordinat_kunjungan']
                
                rute_per_algo_wilayah[id_algo][wilayah_lower_key].append({
                    "id": str(id_kurir),
                    "path": urutan_lokasi,
                    "jarak": jarak_km,
                    "waktu": waktu_menit
                })
                
                class_identitas = f"algo-{id_algo} wil-{wilayah_lower_key} kurir-{id_kurir}"
                
                if id_kurir % 2 != 0:
                    warna_rute = warna_wilayah.get(wilayah_lower_key, "#bdc3c7")
                else:
                    warna_rute = warna_kurir_2.get(wilayah_lower_key, "#7f8c8d")
                
                for i in range(len(urutan_lokasi) - 1):
                    lokasi_asal = urutan_lokasi[i]
                    lokasi_tujuan = urutan_lokasi[i+1]
                    coord_asal = koordinat_lokasi[i]
                    coord_tujuan = koordinat_lokasi[i+1]
                    
                    rute_nyata = get_real_route_osrm(coord_asal, coord_tujuan)
                    
                    folium.PolyLine(
                        locations=rute_nyata,
                        color=warna_rute,
                        weight=5,
                        opacity=0.9,
                        className=class_identitas,
                        tooltip=f"<b>{info['nama']} - Klaster {wilayah}</b><br>Kurir ID: {id_kurir}<br>Jarak: {jarak_km} km | Waktu: {waktu_menit} Menit<br>{lokasi_asal} ➔ {lokasi_tujuan}"
                    ).add_to(peta)
            
                    if "Gudang Farmasi" not in lokasi_tujuan:
                        urutan_nomor = i + 1
                        data_pusk = info_pusk.get(lokasi_tujuan.strip(), {
                            'warna_prioritas': '#bdc3c7', 'label_prioritas': 'Tidak Terdata', 
                            'layanan': '-', 'jaringan': '-'
                        })
                        warna_prioritas = data_pusk['warna_prioritas']
                        
                        icon_html = f"""
                        <div style="
                            background-color: {warna_prioritas};
                            color: white; width: 22px; height: 22px; border-radius: 50%;
                            text-align: center; line-height: 20px; font-size: 10px;
                            font-weight: bold; border: 1.5px solid white; box-shadow: 0 1px 4px rgba(0,0,0,0.3);
                        ">{urutan_nomor}</div>
                        """
                        
                        popup_html = f"""
                        <div style="font-family: Arial, sans-serif; min-width: 190px; font-size: 12px;">
                            <b style="color:{warna_prioritas}; font-size: 13px;">{urutan_nomor}. {lokasi_tujuan}</b>
                            <hr style="margin: 5px 0; border: 0; border-top: 1px solid #eee;">
                            <b>Tingkat Prioritas:</b> {data_pusk['label_prioritas']}<br>
                            <b>Jaringan:</b> {data_pusk['jaringan']}<br>
                            <b>Jenis Layanan:</b> {data_pusk['layanan']}
                            <hr style="margin: 5px 0; border: 0; border-top: 1px solid #eee;">
                            <b>Alokasi Pengantaran:</b> Kurir {id_kurir} ({wilayah})<br>
                            <b>Jarak & Waktu Tempuh Kurir:</b> {jarak_km} km | {waktu_menit} Menit
                        </div>
                        """
                        
                        folium.Marker(
                            location=coord_tujuan,
                            popup=folium.Popup(popup_html, max_width=300),
                            icon=folium.DivIcon(html=icon_html, class_name=class_identitas)
                        ).add_to(peta)
                    
                    # Delay API
                    time.sleep(0.4)

    rute_js = json.dumps(rute_per_algo_wilayah)
    warna_js = json.dumps(warna_wilayah)

    css_dynamic_buttons = ""
    for wil, warna in warna_wilayah.items():
        css_dynamic_buttons += f".btn-wil[data-wil='{wil}']:hover, .btn-wil.active-{wil} {{ background: {warna} !important; color: #ffffff !important; box-shadow: 0 3px 8px {warna}60; }}\n"

    # 5. DASHBOARD SIDEBAR DENGAN TAMBAHAN LEGEND & PERUBAHAN CSS TOMBOL ALGO
    custom_dashboard = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap');
        
        .custom-dashboard {{
            position: absolute; top: 15px; left: 15px; z-index: 9999;
            background: rgba(255, 255, 255, 0.96);
            backdrop-filter: blur(8px);
            padding: 15px 18px;
            border-radius: 14px;
            box-shadow: 0px 8px 25px rgba(0, 0, 0, 0.15);
            font-family: 'Poppins', sans-serif; color: #2d3436; 
            width: 360px; border: 1px solid rgba(0,0,0,0.08);
            max-height: 92vh; display: flex; flex-direction: column;
        }}
        
        .dash-header {{ flex-shrink: 0; }}
        
        .dash-title {{ 
            font-size: 14px; font-weight: 700; margin: 0 0 10px 0; 
            letter-spacing: 0.5px; text-transform: uppercase; color: #2d3436; 
            border-bottom: 2px solid #f1f2f6; padding-bottom: 8px;
        }}
        
        .section-label {{ font-size: 10px; font-weight: 600; color: #636e72; margin-bottom: 5px; text-transform: uppercase; letter-spacing: 1px; }}
        .btn-container {{ display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 10px; }}
        
        .btn-pill {{
            padding: 5px 12px; border-radius: 20px; border: 1.5px solid transparent; font-size: 11px; font-weight: 600;
            cursor: pointer; transition: all 0.2s ease; background: #f1f2f6; color: #636e72; 
            font-family: 'Poppins', sans-serif;
        }}
        
        /* Modifikasi UI Tombol Algoritma (Abu-abu & Border Hitam) */
        .btn-algo:hover {{ background: #dfe6e9; color: #2d3436; }}
        .btn-algo.active {{ 
            background: #e2e6ea !important; 
            color: #2d3436 !important; 
            border-color: #2d3436 !important; 
            box-shadow: 0 2px 4px rgba(0,0,0,0.15); 
        }}
        
        #btn-semua-wil {{ margin-bottom: 10px; width: 100%; transition: all 0.2s ease; padding: 6px; }}
        
        {css_dynamic_buttons}

        .btn-kurir {{ padding: 5px 12px; cursor: pointer; background: #eee; border: none; border-radius: 5px; font-size: 11px; font-weight: 600; flex: 1; transition: all 0.2s; }}
        .btn-kurir.active {{ background: #2d3436; color: white; box-shadow: 0 2px 5px rgba(0,0,0,0.2); }}

        .metrics-container {{ display: flex; gap: 8px; margin-top: 5px; margin-bottom: 10px; }}
        .metric-box {{ background: #f8f9fa; border-radius: 8px; padding: 8px; text-align: center; border: 1px solid #e9ecef; flex: 1; }}
        .metric-label {{ font-size: 9px; color: #636e72; font-weight: 600; display: block; }}
        .metric-value {{ font-size: 18px; font-weight: 700; color: #2d3436; line-height: 1.1; display: inline-block; margin-top: 3px; }}
        .metric-unit {{ font-size: 11px; font-weight: 600; color: #b2bec3; }}

        .route-list-container {{ margin-top: 8px; overflow-y: auto; flex-grow: 1; padding-right: 4px; display: flex; flex-direction: column; gap: 8px; }}
        .route-list-container::-webkit-scrollbar {{ width: 5px; }}
        .route-list-container::-webkit-scrollbar-track {{ background: #f1f2f6; border-radius: 10px; }}
        .route-list-container::-webkit-scrollbar-thumb {{ background: #b2bec3; border-radius: 10px; }}
        
        .route-group {{ background: #ffffff; border-radius: 8px; padding: 8px 10px; border-left: 4px solid; border-top: 1px solid #f1f2f6; border-right: 1px solid #f1f2f6; border-bottom: 1px solid #f1f2f6; box-shadow: 0 1px 3px rgba(0,0,0,0.02); }}
        .route-group-title {{ font-size: 11px; font-weight: 700; text-transform: uppercase; margin-bottom: 6px; color: #2d3436; }}
        .route-steps {{ list-style: none; padding: 0; margin: 0; position: relative; }}
        .route-steps::before {{ content: ''; position: absolute; top: 10px; bottom: 10px; left: 6px; width: 2px; background: #dfe6e9; z-index: 1; }}
        .route-step-item {{ font-size: 11px; padding: 3px 0 3px 20px; position: relative; color: #636e72; line-height: 1.3; }}
        .route-step-item::before {{ content: ''; position: absolute; left: 2px; top: 6px; width: 10px; height: 10px; border-radius: 50%; background: #ffffff; border: 2.5px solid; z-index: 2; border-color: inherit; }}

        .hide-ga .algo-ga {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-ma .algo-ma {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-aco .algo-aco {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-pso .algo-pso {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        
        .hide-pusat .wil-pusat {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-utara .wil-utara {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-selatan .wil-selatan {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-timur .wil-timur {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-barat .wil-barat {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        
        .hide-kurir-1 .kurir-1 {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-kurir-2 .kurir-2 {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-kurir-3 .kurir-3 {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
        .hide-kurir-4 .kurir-4 {{ display: none !important; opacity: 0 !important; pointer-events: none !important; }}
    </style>

    <div class="custom-dashboard" id="mainDashboard">
        <div class="dash-header">
            <h3 class="dash-title">📊 Rute Distribusi Farmasi</h3>
            
            <div class="section-label">Pilih Algoritma</div>
            <div class="btn-container">
                <button class="btn-pill btn-algo active" data-algo="ga">GA</button>
                <button class="btn-pill btn-algo" data-algo="ma">MA</button>
                <button class="btn-pill btn-algo" data-algo="aco">ACO</button>
                <button class="btn-pill btn-algo" data-algo="pso">PSO</button>
            </div>
            
            <div class="section-label" style="margin-top: 8px;">Prioritas Kunjungan (Legend)</div>
            <div style="font-size: 10px; margin-bottom: 12px; display: flex; flex-direction: column; gap: 4px; color: #636e72; font-weight: 500;">
                <div style="display: flex; align-items: center;"><span style="width:10px; height:10px; border-radius:50%; background-color:#e74c3c; margin-right:6px; border:1px solid #c0392b;"></span> 1: Induk - Rawat Inap (Sangat Mendesak)</div>
                <div style="display: flex; align-items: center;"><span style="width:10px; height:10px; border-radius:50%; background-color:#e67e22; margin-right:6px; border:1px solid #d35400;"></span> 2: Induk - Rawat Jalan (Mendesak)</div>
                <div style="display: flex; align-items: center;"><span style="width:10px; height:10px; border-radius:50%; background-color:#3498db; margin-right:6px; border:1px solid #2980b9;"></span> 3: Pustu - Rawat Jalan (Reguler)</div>
            </div>
            
            <div class="section-label">Filter Wilayah (Klaster)</div>
            <button class="btn-pill" id="btn-semua-wil"></button>
            <div class="btn-container">
                <button class="btn-pill btn-wil active-pusat" data-wil="pusat">Pusat</button>
                <button class="btn-pill btn-wil active-utara" data-wil="utara">Utara</button>
                <button class="btn-pill btn-wil active-selatan" data-wil="selatan">Selatan</button>
                <button class="btn-pill btn-wil active-timur" data-wil="timur">Timur</button>
                <button class="btn-pill btn-wil active-barat" data-wil="barat">Barat</button>
            </div>

            <div id="filterKurirSection">
                <div class="section-label">Filter Kurir & Validasi Waktu</div>
                <div class="btn-container" style="display: flex;">
                    <button class="btn-kurir active" data-kurir="1">Kurir 1</button>
                    <button class="btn-kurir active" data-kurir="2">Kurir 2</button>
                </div>
            </div>
            
            <div class="metrics-container">
                <div class="metric-box">
                    <span class="metric-label">ESTIMASI JARAK</span>
                    <div><span class="metric-value" id="distDisplay">0.00</span> <span class="metric-unit">km</span></div>
                </div>
                <div class="metric-box">
                    <span class="metric-label">TOTAL WAKTU</span>
                    <div><span class="metric-value" id="timeDisplay">0.00</span> <span class="metric-unit">mnt</span></div>
                </div>
            </div>
            
            <div class="section-label">Urutan Rute Kunjungan</div>
        </div>
        
        <div class="route-list-container" id="routeListContainer">
            </div>
    </div>

    <script>
    setTimeout(function() {{
        const mapContainer = document.querySelector('.leaflet-container');
        const dataRute = {rute_js};
        const warnaWilayah = {warna_js};
        
        const algoBtns = document.querySelectorAll('.btn-algo');
        const wilBtns = document.querySelectorAll('.btn-wil');
        const kurirBtns = document.querySelectorAll('.btn-kurir');
        const btnSemuaWil = document.getElementById('btn-semua-wil');
        const distDisplay = document.getElementById('distDisplay');
        const timeDisplay = document.getElementById('timeDisplay');
        const routeListContainer = document.getElementById('routeListContainer');
        const filterKurirSection = document.getElementById('filterKurirSection');
        
        let activeAlgo = "ga";
        let activeWils = new Set(["pusat", "utara", "selatan", "timur", "barat"]);
        
        function updateSelectAllBtn() {{
            if (activeWils.size > 0) {{
                btnSemuaWil.innerText = "Hapus Semua Pilihan";
                btnSemuaWil.style.background = "#ff7675"; btnSemuaWil.style.color = "#ffffff";
            }} else {{
                btnSemuaWil.innerText = "Pilih Semua Wilayah";
                btnSemuaWil.style.background = "#74b9ff"; btnSemuaWil.style.color = "#ffffff";
            }}
        }}
        
        function generateRouteListUI(activeKurirIds) {{
            routeListContainer.innerHTML = ""; 
            if (!dataRute[activeAlgo]) return;
            
            const urutanWilayah = ["pusat", "utara", "selatan", "timur", "barat"];
            
            urutanWilayah.forEach(wil => {{
                if (activeWils.has(wil) && dataRute[activeAlgo][wil]) {{
                    const listKurir = dataRute[activeAlgo][wil];
                    const warna = warnaWilayah[wil] || "#bdc3c7";
                    
                    listKurir.forEach((kurirData) => {{
                        if (!activeKurirIds.includes(kurirData.id.toString())) return;

                        const groupDiv = document.createElement('div');
                        groupDiv.className = 'route-group';
                        groupDiv.style.borderLeftColor = warna;
                        groupDiv.style.marginBottom = '8px';
                        
                        const titleDiv = document.createElement('div');
                        titleDiv.className = 'route-group-title';
                        
                        const infoText = `${{kurirData.jarak}} km | ${{kurirData.waktu}} mnt`;
                        titleDiv.innerHTML = `<strong>Klaster ${{wil.charAt(0).toUpperCase() + wil.slice(1)}}</strong> 
                                              <span style="float: right; color: #0984e3; font-weight:600;">Kurir ${{kurirData.id}}</span>
                                              <div style="font-size:9px; color:#b2bec3; font-weight:500; text-transform:none; margin-top:2px;">${{infoText}}</div>`;
                        
                        groupDiv.appendChild(titleDiv);
                        
                        const ulElement = document.createElement('ul');
                        ulElement.className = 'route-steps';
                        
                        kurirData.path.forEach((titik, index) => {{
                            const liElement = document.createElement('li');
                            liElement.className = 'route-step-item';
                            const isGudang = titik.includes("Gudang Farmasi");
                            let textPuskes = titik.replace("Surabaya", "").trim();
                            
                            liElement.innerText = isGudang ? textPuskes : `${{index}}. ${{textPuskes}}`;
                            liElement.style.borderColor = warna;
                            ulElement.appendChild(liElement);
                        }});
                        
                        groupDiv.appendChild(ulElement);
                        routeListContainer.appendChild(groupDiv);
                    }});
                }}
            }});
        }}
        
        function updateDashboard() {{
            ['ga', 'ma', 'aco', 'pso'].forEach(a => {{
                if(a === activeAlgo) mapContainer.classList.remove('hide-' + a);
                else mapContainer.classList.add('hide-' + a);
            }});
            ['pusat', 'utara', 'selatan', 'timur', 'barat'].forEach(w => {{
                if(activeWils.has(w)) mapContainer.classList.remove('hide-' + w);
                else mapContainer.classList.add('hide-' + w);
            }});
            
            let uniqueKurirs = new Set();
            activeWils.forEach(w => {{
                if(dataRute[activeAlgo] && dataRute[activeAlgo][w]) {{
                    dataRute[activeAlgo][w].forEach(k => {{
                        uniqueKurirs.add(k.id.toString());
                    }});
                }}
            }});

            if(uniqueKurirs.size <= 1) {{
                filterKurirSection.style.display = 'none';
            }} else {{
                filterKurirSection.style.display = 'block';
            }}

            let activeKurirIds = Array.from(document.querySelectorAll('.btn-kurir.active')).map(b => b.getAttribute('data-kurir'));
            
            if(uniqueKurirs.size <= 1) {{
                activeKurirIds = Array.from(uniqueKurirs);
            }}

            ['1', '2', '3', '4'].forEach(k => {{
                if (activeKurirIds.includes(k)) mapContainer.classList.remove('hide-kurir-' + k);
                else mapContainer.classList.add('hide-kurir-' + k);
            }});
            
            let totalKm = 0;
            let totalMenit = 0;
            
            activeWils.forEach(w => {{
                if(dataRute[activeAlgo] && dataRute[activeAlgo][w]) {{
                    dataRute[activeAlgo][w].forEach(kurirData => {{
                        if (activeKurirIds.includes(kurirData.id.toString())) {{
                            totalKm += kurirData.jarak || 0;
                            totalMenit += kurirData.waktu || 0;
                        }}
                    }});
                }}
            }});
            
            distDisplay.innerText = totalKm.toFixed(2);
            timeDisplay.innerText = totalMenit.toFixed(2);
            
            generateRouteListUI(activeKurirIds);
        }}
        
        algoBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                algoBtns.forEach(b => b.classList.remove('active'));
                this.classList.add('active');
                activeAlgo = this.getAttribute('data-algo');
                updateDashboard();
            }});
        }});
        
        wilBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                const w = this.getAttribute('data-wil');
                const activeClass = 'active-' + w;
                if(activeWils.has(w)) {{
                    activeWils.delete(w);
                    this.classList.remove(activeClass);
                }} else {{
                    activeWils.add(w);
                    this.classList.add(activeClass);
                }}
                updateSelectAllBtn();
                updateDashboard();
            }});
        }});
        
        btnSemuaWil.addEventListener('click', function() {{
            if (activeWils.size > 0) {{
                activeWils.clear();
                wilBtns.forEach(btn => btn.classList.remove('active-' + btn.getAttribute('data-wil')));
            }} else {{
                wilBtns.forEach(btn => {{
                    const w = btn.getAttribute('data-wil');
                    activeWils.add(w);
                    btn.classList.add('active-' + w);
                }});
            }}
            updateSelectAllBtn();
            updateDashboard();
        }});
        
        kurirBtns.forEach(btn => {{
            btn.addEventListener('click', function() {{
                this.classList.toggle('active');
                updateDashboard();
            }});
        }});
        
        mapContainer.classList.add('hide-ma', 'hide-aco', 'hide-pso');
        updateSelectAllBtn();
        updateDashboard();
    }}, 500);
    </script>
    """
    
    peta.get_root().html.add_child(folium.Element(custom_dashboard))
    
    output_html = "visualisasi_perbandingan_rute_EAS.html"
    peta.save(output_html)
    print(f"\nSukses! File visualisasi mandiri '{output_html}' berhasil dibuat.")
    return peta

peta_final = buat_peta_rute_eas()
peta_final

Memulai pembacaan seluruh JSON hasil algoritma dan penarikan jalan OSRM. Mohon tunggu, proses ini memakan waktu agar server peta tidak memblokir koneksi...


KeyboardInterrupt: 